# DeiT-Tiny seed5 — E→O raw top1 vs Vanilla + rollback

Attach CIFAR-100; enable Internet, T4x2 and Kaggle Secret `github_token`.
Optional resume: full output archive from this same seed/recipe. Compatible
seed5 E→O progress from the earlier raw-vs-Bypass notebook can be reused only
with its original hash-bound fork/raw reference and matching configuration.
Each requested arm resumes independently. Controls are E→O raw and Vanilla rollback;
the v2 section below adds explicit research arms, without Bypass or A3/A4.

Phase1 trains Vanilla for seed5 until150 epochs without a strict validation
accuracy best. No minimum epoch and no requirement to finish schedule300.
Reload that historical-best model, AdamW, scheduler and RNG/loader state.

When RUN_CONTINUATIONS=True, Phase2 runs two control trajectories from identical full fork:
- GPU0: `e_driven_o_raw`, original TINY top1 WHERE over12 MLPs, one E→O projection.
- GPU1: `vanilla_rollback`, genuine Vanilla training without E/projection/moment reset.

Both use patience10. Higher accuracy OR an exact accuracy tie with lower loss
updates the controller anchor and resets stall; otherwise stall increments once.
Rollback restores model/optimizer/scheduler and retains the current RNG/loader
stream. No retrigger. Report best uses strict accuracy and paired loss.
Scientific escape compares observed post-fork best to this seed's historical best.
Vanilla rollback is newly trained, not the observed Vanilla plateau-confirmation
window. On resume completed arms skip training and remaining arms use free GPU slots.

Recipe: batch64, AdamW LR5e-4/WD.05, warmup5 + cosine300, plateau150, horizon150,
max_epoch800. E→O rank4, stats256, WHERE3x32, projection32, gate32, CG200,
scales(.0125,.025,.05). Evaluation uses3000 validation samples;2000 reserved
samples do not trigger/select anything. Official test is unused.

Every epoch saves full state including anchor/counter, optimizer, scheduler,
RNG and loader. Download the full output tarball for another session.
The comparison uses completed current arms with identical fork/protocol/horizon;
failed or unrequested results do not enter the accuracy delta.

## Optional Counterfactual Projection v2 research
The notebook prepares the same fork/raw reference, then runs technical diagnostics
before any continuation. TINY, GradMax GELU-SVD plus incoming-only steps, and
full-A/a/B Optimized-E share the fixed original TINY top1 site. Compare absolute
and relative_gram damping. Heldout256 is independent of WHERE/fit/gate/tuning;
margin/nonlinear errors are diagnostic only. Multi-MLP scope is diagnostic only.
SVD range/regularization/solver decomposition uses a small rank-deficient problem;
the full DeiT minimum residual remains unknown. SSD is not registered until its
GELU splitting-matrix assumptions are audited. No accuracy gain is established
by these diagnostics. GradMax source: https://arxiv.org/abs/2201.05125

Default RUN_CONTINUATIONS=False runs Phase1/fork preparation and diagnostics only.
After reviewing technical results, set it True to run the original two controls
and five new one-shot arms (TINY relative, GradMax/Optimized-E absolute/relative),
150 epochs each, shared patience10 rollback, no retrigger. Two GPUs queue arms.
Existing compatible control results are reused. v2 resume requires the original
generator_reference.pt plus full arm latest; download the full output archive.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO = Path("/kaggle/working/deit-all-arms-repo")
OUTPUT = Path("/kaggle/working/projection_log_analysis")
URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
token = UserSecretsClient().get_secret("github_token").strip()
if not token:
    raise RuntimeError("Enable the github_token Kaggle Secret")
askpass = Path("/kaggle/working/.projection_log_git_askpass.py")
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy()
env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
try:
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", URL, str(REPO)], env=env, check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], env=env, check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], env=env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN_RUNTIME", None)
    del token
GROMO = Path("/kaggle/working/deit-gromo")
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
if not GROMO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                    "https://github.com/growingnet/gromo.git", str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1", "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach", GROMO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[vision,test]", "-e", str(GROMO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join((str(REPO), str(GROMO / "src")))
ENV["REQUIRE_DEIT_INTEGRATION"] = "1"
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_deit_function_preserving.py", "tests/test_deit_sites.py",
    "tests/test_deit_projection.py", "tests/test_deit_no_aux_persistence.py",
    "tests/test_deit_protocol.py", "tests/test_deit_vanilla_reference.py", "tests/test_deit_e_rollback.py", "tests/test_deit_ablation.py", "tests/test_deit_ablation_integration.py", "tests/test_deit_logging.py", "tests/test_deit_parallel.py", "tests/test_deit_resume.py", "tests/test_research_projection_v2.py", "tests/test_kaggle_checkpoint_discovery.py"], cwd=REPO, env=ENV, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable a CUDA GPU; this notebook trains DeiT")
print("Repository commit:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from dataclasses import asdict
from experiments.deit_protocol import DeitRecipe, protocol, canonical_model_config
from adapters.deit_ablation import res18_cp_config
from experiments.shared_protocol import sha256_file

SEED = 5
ARMS = ('e_driven_o_raw', 'vanilla_rollback')
SCHEDULE_EPOCHS = 300
STALL_PATIENCE = 150
POST_FORK_EPOCHS = 150
ALGORITHM_PATIENCE = 10
MAX_EPOCH = 800
BATCH_SIZE = 64
GPU_DEVICES = 'auto'  # T4x2: two workers; one GPU: one worker.
LEARNING_RATE = 5e-4  # Explicit per-arm LR; no implicit ImageNet batch scaling.
RANK = 4
PROJECTION_SAMPLES = 32
SCALES = '.0125,.025,.05'
OUTPUT = Path(f'/kaggle/working/deit_tiny_seed{SEED}_raw_vs_vanilla_rollback_v1')
OUTPUT.mkdir(parents=True, exist_ok=True)
RECIPE = DeitRecipe(seed=SEED, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    schedule_epochs=SCHEDULE_EPOCHS, stall_start_epoch=0,
    stall_patience=STALL_PATIENCE, reference_epochs=POST_FORK_EPOCHS, max_epoch=MAX_EPOCH)
RECIPE.validate()
EXPECTED = protocol(RECIPE, canonical_model_config())
CP = res18_cp_config()
from dataclasses import replace
CP = replace(CP, rank=RANK, projection_samples=PROJECTION_SAMPLES,
             scales=tuple(map(float, SCALES.split(','))))
roots = set()
for directory, _, files in os.walk('/kaggle/input', followlinks=True):
    if Path(directory).name == 'cifar-100-python' and {'train', 'meta'}.issubset(files):
        roots.add(Path(directory).parent.resolve())
if len(roots) != 1:
    raise FileNotFoundError(f'Attach exactly one CIFAR-100 dataset: {roots}')
DATA_ROOT = next(iter(roots))
print('Recipe:', asdict(RECIPE))
print('Projection:', asdict(CP))

RUN_CONTINUATIONS = False
RESEARCH = OUTPUT / 'research_v2'


In [ ]:
# Import full outputs (original .pt, .tar.gz/.zip, or mounted notebook output).
import shutil
from experiments.deit_resume import prepare_resume
RESUME_PLAN = prepare_resume(['/kaggle/input'], OUTPUT, RECIPE, CP,
    horizon=POST_FORK_EPOCHS, patience=ALGORITHM_PATIENCE, inner_steps=5, arms=ARMS)
PHASE1 = OUTPUT / 'vanilla_stall'
FORK = PHASE1 / 'plateau_checkpoint.pt'
VANILLA_REFERENCE = PHASE1 / 'vanilla_reference.pt'


In [ ]:
command = [sys.executable, '-m', 'experiments.run_deit_all_arms', '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT), '--seed', str(SEED), '--schedule-epochs', str(SCHEDULE_EPOCHS),
    '--max-epoch', str(MAX_EPOCH), '--batch-size', str(BATCH_SIZE), '--learning-rate', str(LEARNING_RATE),
    '--stall-patience', str(STALL_PATIENCE), '--post-fork-epochs', str(POST_FORK_EPOCHS),
    '--algorithm-patience', str(ALGORITHM_PATIENCE), '--rank', str(RANK),
    '--projection-samples', str(PROJECTION_SAMPLES), '--scales', SCALES, '--opt-inner-steps', '5',
    '--gpu-devices', GPU_DEVICES, '--arms', ','.join(ARMS)]
command += ['--prepare-only']
if FORK.exists():
    command += ['--plateau-checkpoint', str(FORK), '--vanilla-reference', str(VANILLA_REFERENCE)]
try:
    subprocess.run(command, cwd=REPO, env=ENV, check=True)
finally:
    archive = shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT)
    print('Resumable output archive:', archive)


In [ ]:
research_command = [sys.executable, '-m', 'experiments.run_deit_research_v2',
    '--data-root', str(DATA_ROOT), '--plateau-checkpoint', str(FORK),
    '--raw-reference', str(OUTPUT / 'raw_intervention_reference.pt'),
    '--output', str(RESEARCH), '--input-root', '/kaggle/input']
try:
    subprocess.run(research_command, cwd=REPO, env=ENV, check=True)
finally:
    print('Resumable output archive:', shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT))
print((RESEARCH / 'diagnostics.json').read_text())


In [ ]:
if RUN_CONTINUATIONS:
    try:
        baseline_command = [value for value in command if value != '--prepare-only']
        subprocess.run(baseline_command, cwd=REPO, env=ENV, check=True)
        subprocess.run(research_command + ['--phase', 'all-continuations', '--gpu-devices', GPU_DEVICES],
                       cwd=REPO, env=ENV, check=True)
    finally:
        print('Resumable output archive:', shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT))
else:
    print('Technical diagnostics only; set RUN_CONTINUATIONS=True after reviewing the report.')


In [ ]:
if RUN_CONTINUATIONS:
    from experiments.shared_protocol import atomic_json_save
    status_path = OUTPUT / 'arm_status.json'
    statuses = json.loads(status_path.read_text()) if status_path.exists() else {}
    comparison = {}
    expected_hash = sha256_file(FORK)
    for method in ARMS:
        status = statuses.get(method, {})
        if status.get('status') != 'completed':
            comparison[method] = {'available': False, 'status': status.get('status', 'not_completed')}
            continue
        result = json.loads((OUTPUT / 'arms' / method / 'result.json').read_text())
        identity = result['run_identity']
        if (result['theta_best_hash'] != expected_hash or result['protocol'] != EXPECTED
                or identity['cp_config'] != json.loads(json.dumps(asdict(CP)))
                or identity['algorithm_patience'] != ALGORITHM_PATIENCE
                or identity['post_fork_epochs'] != POST_FORK_EPOCHS
                or result['history'][-1]['post_fork_epoch'] != POST_FORK_EPOCHS):
            raise ValueError('Seed5 comparison requires the same complete fork/protocol/config/horizon')
        comparison[method] = {'available': True, **{key: result[key] for key in (
            'fork_epoch', 'historical_best_accuracy', 'report_best_accuracy',
            'report_best_loss', 'report_best_epoch', 'delta_vs_historical_best', 'scientific_escape')},
            'rollback_count': len(result['rollback_events']), 'fork_hash': expected_hash}
    raw = comparison.get('e_driven_o_raw', {})
    vanilla = comparison.get('vanilla_rollback', {})
    eligible = raw.get('available', False) and vanilla.get('available', False)
    report = {'arms': comparison, 'comparison_eligible': eligible,
        'e2o_minus_vanilla_rollback_best_accuracy':
            raw['report_best_accuracy'] - vanilla['report_best_accuracy'] if eligible else None}
    atomic_json_save(report, OUTPUT / 'raw_vs_vanilla_rollback_comparison.json')
    print(json.dumps(report, indent=2))
    print('Status:', statuses)
    print((RESEARCH / 'continuation_comparison.json').read_text())


## Final summary
Seven continuation arms, including incomplete/failed arms. Accuracy is percent; deltas are percentage points.
Only complete results matching the current fork/protocol/config enter comparisons. Diagnostic previews are shown separately.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

def read_json_if_present(path):
    return json.loads(path.read_text()) if path.exists() else {}

arm_specs = [('vanilla_rollback', 'Vanilla + rollback'),
             ('e_driven_o_raw', 'TINY / absolute'),
             ('cp_v2_tiny_relative_gram', 'TINY / relative_gram'),
             ('cp_v2_gradmax_absolute', 'GradMax / absolute'),
             ('cp_v2_gradmax_relative_gram', 'GradMax / relative_gram'),
             ('cp_v2_optimized_e_absolute', 'Optimized-E / absolute'),
             ('cp_v2_optimized_e_relative_gram', 'Optimized-E / relative_gram')]
statuses = {**read_json_if_present(OUTPUT / 'arm_status.json'),
            **read_json_if_present(RESEARCH / 'v2_arm_status.json')}
fork_hash = sha256_file(FORK) if FORK.exists() else None
reference_path = RESEARCH / 'generator_reference.pt'
reference_hash = sha256_file(reference_path) if reference_path.exists() else None
summary_rows = []
completed_results = {}
for method, label in arm_specs:
    root = RESEARCH if method.startswith('cp_v2_') else OUTPUT / 'arms'
    row = {'Arm': label, 'Status': statuses.get(method, {}).get('status', 'not_started')}
    try:
        result = read_json_if_present(root / method / 'result.json')
        if result:
            identity = result['run_identity']
            expected_cp = json.loads(json.dumps(asdict(CP)))
            if method.startswith('cp_v2_'):
                expected_cp.update(damping_mode='absolute' if method.endswith('_absolute')
                    else 'relative_gram', trace_probes=8, trace_seed=2028)
                if reference_hash is None or identity['generator_reference_hash'] != reference_hash:
                    raise ValueError('generator reference mismatch')
            if (result['theta_best_hash'] != fork_hash or result['protocol'] != EXPECTED
                    or identity['cp_config'] != expected_cp
                    or identity['post_fork_epochs'] != POST_FORK_EPOCHS
                    or identity['algorithm_patience'] != ALGORITHM_PATIENCE
                    or result['history'][-1]['post_fork_epoch'] != POST_FORK_EPOCHS):
                raise ValueError('fork/protocol/config/horizon mismatch')
            row.update(Status='completed', **{'Best acc (%)': 100 * result['report_best_accuracy'],
                'Best loss': result['report_best_loss'], 'Best epoch': result['report_best_epoch'],
                'Delta vs fork (pp)': 100 * result['delta_vs_historical_best'],
                'Scientific escape': result['scientific_escape'], 'Rollbacks': len(result['rollback_events'])})
            completed_results[method] = (label, result)
        elif row['Status'] == 'completed':
            row['Status'] = 'missing_result'
    except (KeyError, ValueError, TypeError, IndexError) as error:
        row.update(Status='excluded', Reason=str(error))
    summary_rows.append(row)
vanilla = next((row['Best acc (%)'] for row in summary_rows
                if row['Arm'] == 'Vanilla + rollback' and row['Status'] == 'completed'), None)
for row in summary_rows:
    row['Delta vs Vanilla (pp)'] = (row['Best acc (%)'] - vanilla
        if row['Status'] == 'completed' and vanilla is not None else None)
columns = ['Arm', 'Status', 'Best acc (%)', 'Best loss', 'Best epoch',
           'Delta vs fork (pp)', 'Delta vs Vanilla (pp)', 'Scientific escape', 'Rollbacks', 'Reason']
summary_table = pd.DataFrame(summary_rows).reindex(columns=columns)
summary_table.to_csv(OUTPUT / 'final_arm_summary.csv', index=False)
display(Markdown('### Continuation results'))
display(summary_table.style.format({'Best acc (%)': '{:.3f}', 'Best loss': '{:.5f}',
    'Best epoch': '{:.0f}', 'Delta vs fork (pp)': '{:+.3f}',
    'Delta vs Vanilla (pp)': '{:+.3f}', 'Rollbacks': '{:.0f}'}, na_rep='—'))

diagnostic = read_json_if_present(RESEARCH / 'diagnostics.json')
if diagnostic.get('rows'):
    diagnostic_table = pd.DataFrame(diagnostic['rows']).reindex(columns=[
        'generator', 'selected_site', 'projection_scope', 'damping_mode', 'cg_converged',
        'e_gain', 'projection_relative_residual', 'heldout_projection_residual',
        'heldout_cosine', 'gate_gain', 'gate_accepted'])
    display(Markdown('### Projection diagnostics — previews, not continuation accuracy'))
    display(diagnostic_table)
print('Saved:', OUTPUT / 'final_arm_summary.csv')
print('Resumable output archive:', shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT))


## Rollback trajectories
All seven arms use rollback. Graphs use the complete compatible results accepted by the summary cell.
The x-axis counts elapsed post-fork training epochs 1–150 and never rewinds. Solid lines show validation measured before the controller; dashed lines show the stored anchor.
Red markers/arrows show the restored anchor at each rollback, not a fresh validation evaluation. Labels identify the original anchor epoch.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plot_root = OUTPUT / 'plots'
plot_root.mkdir(parents=True, exist_ok=True)
if not completed_results:
    print('No complete compatible continuation results yet; no rollback curves to plot.')
else:
    # Overlay measured accuracy/loss for a direct comparison of the arms.
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
    for method, (label, result) in completed_results.items():
        history = pd.DataFrame(result['history']).sort_values('post_fork_epoch').query('post_fork_epoch > 0')
        x = history['post_fork_epoch']
        axes[0].plot(x, 100 * history['validation_accuracy'], label=label, linewidth=1.2)
        axes[1].plot(x, history['validation_loss'], label=label, linewidth=1.2)
    reference = next(iter(completed_results.values()))[1]
    axes[0].axhline(100 * reference['historical_best_accuracy'], color='black',
                   linestyle='--', linewidth=1, label='Historical-best fork')
    axes[1].axhline(reference['historical_best_loss'], color='black', linestyle='--', linewidth=1)
    for ax, title, ylabel in zip(axes, ('Validation accuracy comparison', 'Validation loss comparison'),
                                 ('Validation accuracy (%)', 'Validation loss')):
        ax.set(title=title, xlabel='Elapsed post-fork epochs', ylabel=ylabel)
        ax.grid(alpha=.2)
    axes[0].legend(fontsize=8, ncol=2)
    fig.savefig(plot_root / 'all_arms_validation.png', dpi=180)
    fig.savefig(plot_root / 'all_arms_validation.pdf')
    plt.show()
    plt.close(fig)

    for method, (label, result) in completed_results.items():
        history = pd.DataFrame(result['history']).sort_values('post_fork_epoch').query('post_fork_epoch > 0')
        rollback_rows = history.loc[history.get('rollback_applied',
            pd.Series(False, index=history.index)).fillna(False).astype(bool)]
        fig, axes = plt.subplots(1, 2, figsize=(15, 4), constrained_layout=True)
        for ax, metric, anchor_metric, factor, ylabel in (
                (axes[0], 'validation_accuracy', 'controller_anchor_accuracy', 100, 'Validation accuracy (%)'),
                (axes[1], 'validation_loss', 'controller_anchor_loss', 1, 'Validation loss')):
            ax.plot(history['post_fork_epoch'], factor * history[metric], color='tab:blue',
                    linewidth=1.3, label='Measured before controller')
            if anchor_metric in history:
                ax.step(history['post_fork_epoch'], factor * history[anchor_metric], where='post',
                        color='tab:orange', linestyle='--', linewidth=1.2, label='Stored controller anchor')
            for _, row in rollback_rows.iterrows():
                epoch = row['post_fork_epoch']
                observed, restored = factor * row[metric], factor * row[anchor_metric]
                ax.axvline(epoch, color='tab:red', linestyle=':', alpha=.35, linewidth=.8)
                ax.scatter([epoch], [restored], marker='v', color='tab:red', s=30, zorder=4)
                ax.annotate('', xy=(epoch, restored), xytext=(epoch, observed),
                            arrowprops={'arrowstyle':'->', 'color':'tab:red', 'lw':.8})
                if ax is axes[0]:
                    ax.annotate('anchor ' + str(int(row['rollback_anchor_epoch'])),
                                xy=(epoch, restored), xytext=(3, 8), textcoords='offset points',
                                fontsize=7, rotation=90, color='tab:red')
            ax.set(xlabel='Elapsed post-fork epochs', ylabel=ylabel)
            ax.grid(alpha=.2)
        handles, labels = axes[0].get_legend_handles_labels()
        if not rollback_rows.empty:
            handles.append(Line2D([], [], color='tab:red', marker='v', linestyle=':', label='Rollback to anchor'))
            labels.append('Rollback to anchor')
        axes[0].legend(handles, labels, fontsize=8)
        fig.suptitle(f'{label} — {len(rollback_rows)} rollbacks')
        fig.savefig(plot_root / f'{method}_rollback.png', dpi=180)
        fig.savefig(plot_root / f'{method}_rollback.pdf')
        plt.show()
        plt.close(fig)
    print('Saved PNG/PDF graphs:', plot_root)
print('Resumable output archive:', shutil.make_archive(str(OUTPUT), 'gztar', root_dir=OUTPUT))
